In [62]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [63]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


In [64]:
abnb = pd.read_csv("data/ABNB.csv")
tsa = pd.read_csv("data/tsa_2025.csv")
cpi = pd.read_csv("data/fred_cpi_lodging_away_from_home.csv")

In [65]:
abnb.head()
 

,date,ticker,volume,open,close,high,low,window_start,transactions
0,2025-01-02,ABNB,2606269,131.87,131.48,134.230,130.4100,1735794000000000000,51821
1,2025-01-03,ABNB,3607355,131.98,135.71,136.360,131.9400,1735880400000000000,62530
2,2025-01-06,ABNB,4174471,137.00,135.20,138.100,134.5800,1736139600000000000,69444
3,2025-01-07,ABNB,4290458,135.35,131.29,136.765,130.8096,1736226000000000000,64384
4,2025-01-08,ABNB,3204678,131.20,130.80,131.630,130.1000,1736312400000000000,51095


In [66]:
tsa.head()

,Date,Numbers
0,1/1/25,"2,317,817"
1,1/2/25,"2,612,162"
2,1/3/25,"2,563,751"
3,1/4/25,"2,543,303"
4,1/5/25,"2,579,257"


In [67]:
cpi.head()

,observation_date,CUSR0000SEHB
0,2025-06-01,182.607
1,2025-07-01,180.824
2,2025-08-01,184.457


In [68]:
# convert Date column to datetime
tsa['Date'] = pd.to_datetime(tsa['Date'])

# filter for summer period
tsa_summer = tsa[
    (tsa['Date'] >= '2025-06-01') &
    (tsa['Date'] <= '2025-08-15')
]

tsa_summer.head()

/var/folders/4k/02060mxx2w7f0qm05q4snwx80000gn/T/ipykernel_98157/2356625188.py:2: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,Date,Numbers
151,2025-06-01,"2,825,055"
152,2025-06-02,"2,642,700"
153,2025-06-03,"2,213,336"
154,2025-06-04,"2,364,375"
155,2025-06-05,"2,798,927"


In [69]:
# convert date
abnb['date'] = pd.to_datetime(abnb['date'])

# filter summer
abnb_summer = abnb[
    (abnb['date'] >= '2025-06-01') &
    (abnb['date'] <= '2025-08-15')
]

abnb_summer.head()

,date,ticker,volume,open,close,high,low,window_start,transactions
102,2025-06-02,ABNB,5223064,128.85,129.62,129.7700,127.985,1748836800000000000,77434
103,2025-06-03,ABNB,5809271,129.50,132.90,133.4900,128.950,1748923200000000000,76850
104,2025-06-04,ABNB,4501382,132.28,133.51,134.8550,131.640,1749009600000000000,74673
105,2025-06-05,ABNB,5859127,133.50,137.29,137.8025,133.500,1749096000000000000,87427
106,2025-06-06,ABNB,9053459,139.18,140.64,143.8799,138.950,1749182400000000000,128282


In [70]:
# convert date
cpi['observation_date'] = pd.to_datetime(cpi['observation_date'])

# filter summer
cpi_summer = cpi[
    (cpi['observation_date'] >= '2025-06-01') &
    (cpi['observation_date'] <= '2025-08-15')
]

cpi_summer.head()

,observation_date,CUSR0000SEHB
0,2025-06-01,182.607
1,2025-07-01,180.824
2,2025-08-01,184.457


In [71]:
### TSA Passenger Volume (Summer 2025)
tsa_summer_chart_v2 = tsa_summer.copy()
tsa_summer_chart_v2['Date'] = pd.to_datetime(tsa_summer_chart_v2['Date'])
tsa_summer_chart_v2['Numbers'] = pd.to_numeric(
    tsa_summer_chart_v2['Numbers'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

tsa_summer_chart_v2 = tsa_summer_chart_v2.sort_values('Date')

fig_tsa_summer = px.line(
    tsa_summer_chart_v2,
    x='Date',
    y='Numbers',
    title='TSA Passenger Volume During the Summer 2025 Travel Season',
    labels={'Date': 'Date', 'Numbers': 'Passenger Volume'}
)

fig_tsa_summer.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_title='Date',
    yaxis_title='Passenger Volume',
    hovermode='x unified'
)

fig_tsa_summer

In [72]:
# Prepare the summer lodging CPI data for a clean line chart
cpi_summer_chart = cpi_summer.copy()
cpi_summer_chart['observation_date'] = pd.to_datetime(cpi_summer_chart['observation_date'])
cpi_summer_chart['CUSR0000SEHB'] = pd.to_numeric(cpi_summer_chart['CUSR0000SEHB'], errors='coerce')

fig = px.line(
    cpi_summer_chart,
    x='observation_date',
    y='CUSR0000SEHB',
    title='Lodging CPI During the Summer 2025 Travel Season',
    labels={'observation_date': 'Date', 'CUSR0000SEHB': 'Lodging CPI'},
    markers=True
)

fig.update_traces(marker=dict(size=11))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_title='Date',
    yaxis_title='Lodging CPI',
    hovermode='x unified',
    xaxis=dict(
        tickmode='array',
        tickvals=pd.to_datetime(['2025-06-01', '2025-07-01', '2025-08-01']),
        ticktext=['June', 'July', 'August']
    )
)

fig

In [73]:
# Airbnb vs Marriott normalized stock performance (Summer 2025)
mar = pd.read_csv('data/mar.csv')
mar['date'] = pd.to_datetime(mar['date'])

mar_summer = mar[
    (mar['date'] >= '2025-06-01') &
    (mar['date'] <= '2025-08-15')
].copy()

abnb_cmp = abnb_summer[['date', 'close']].copy()
abnb_cmp['date'] = pd.to_datetime(abnb_cmp['date'])
abnb_cmp['close'] = pd.to_numeric(abnb_cmp['close'], errors='coerce')
abnb_cmp = abnb_cmp.sort_values('date')

mar_cmp = mar_summer[['date', 'close']].copy()
mar_cmp['close'] = pd.to_numeric(mar_cmp['close'], errors='coerce')
mar_cmp = mar_cmp.sort_values('date')

abnb_base = abnb_cmp['close'].dropna().iloc[0]
mar_base = mar_cmp['close'].dropna().iloc[0]

abnb_cmp['normalized_index'] = (abnb_cmp['close'] / abnb_base) * 100
mar_cmp['normalized_index'] = (mar_cmp['close'] / mar_base) * 100

abnb_cmp['Company'] = 'Airbnb'
mar_cmp['Company'] = 'Marriott'

comparison_df = pd.concat([
    abnb_cmp[['date', 'normalized_index', 'Company']],
    mar_cmp[['date', 'normalized_index', 'Company']]
], ignore_index=True).sort_values(['Company', 'date'])

fig_abnb_mar = px.line(
    comparison_df,
    x='date',
    y='normalized_index',
    color='Company',
    title='Airbnb vs Marriott Stock Performance (Summer 2025)',
    labels={
        'date': 'Date',
        'normalized_index': 'Relative Stock Performance Index',
        'Company': 'Company'
    },
    color_discrete_map={'Airbnb': '#1f77b4', 'Marriott': '#ff7f0e'},
    markers=True
)

fig_abnb_mar.update_traces(
    mode='lines+markers',
    marker=dict(size=6),
    hovertemplate='Date: %{x|%b %d, %Y}<br>Company: %{fullData.name}<br>Index: %{y:.2f}<extra></extra>'
)

fig_abnb_mar.update_layout(
    template='plotly_white',
    title=dict(
        text='Airbnb vs Marriott Stock Performance (Summer 2025)<br><sup>Indexed to 100 at the start of June 2025 for comparison</sup>',
        x=0.5
    ),
    xaxis_title='Date',
    yaxis_title='Relative Stock Performance Index',
    hovermode='x unified',
    legend_title='Company'
)

fig_abnb_mar.update_xaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)')
fig_abnb_mar.update_yaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)')

fig_abnb_mar

In [74]:
import numpy as np

abnb_scatter = abnb_summer[['date', 'close']].copy()
abnb_scatter['date'] = pd.to_datetime(abnb_scatter['date'])
abnb_scatter['close'] = pd.to_numeric(abnb_scatter['close'], errors='coerce')

tsa_scatter = tsa_summer[['Date', 'Numbers']].copy()
tsa_scatter['date'] = pd.to_datetime(tsa_scatter['Date'])
tsa_scatter['Numbers'] = pd.to_numeric(
    tsa_scatter['Numbers'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

scatter_df = pd.merge(
    abnb_scatter[['date', 'close']],
    tsa_scatter[['date', 'Numbers']],
    on='date',
    how='inner'
).dropna()

# OLS fit: close = intercept + slope * Numbers
X = np.column_stack([np.ones(len(scatter_df)), scatter_df['Numbers'].values])
y = scatter_df['close'].values
beta = np.linalg.lstsq(X, y, rcond=None)[0]
intercept, slope = beta[0], beta[1]

scatter_df = scatter_df.sort_values('Numbers')
scatter_df['ols_trend'] = intercept + slope * scatter_df['Numbers']

fig_abnb_tsa_ols = go.Figure()

fig_abnb_tsa_ols.add_trace(go.Scatter(
    x=scatter_df['Numbers'],
    y=scatter_df['close'],
    mode='markers',
    name='Observed',
    marker=dict(size=9, color='#1F77B4', opacity=0.75),
    hovertemplate='TSA Passengers: %{x:,.0f}<br>ABNB Close: $%{y:.2f}<extra></extra>'
))

fig_abnb_tsa_ols.add_trace(go.Scatter(
    x=scatter_df['Numbers'],
    y=scatter_df['ols_trend'],
    mode='lines',
    name='OLS Trend Line',
    line=dict(color='#D62728', width=3),
    hovertemplate='TSA Passengers: %{x:,.0f}<br>OLS Predicted ABNB: $%{y:.2f}<extra></extra>'
))

fig_abnb_tsa_ols.update_layout(
    title=f'Summer 2025: Airbnb Closing Price vs TSA Passenger Volume',
    template='plotly_white',
    xaxis_title='TSA Passenger Volume',
    yaxis_title='ABNB Closing Price (USD)',
    legend_title='Series'
)

fig_abnb_tsa_ols

In [75]:
# Correlation heatmap: Airbnb, Marriott, Hilton, TSA, and CPI (Summer 2025)
mar = pd.read_csv('data/mar.csv')
hlt = pd.read_csv('data/hlt.csv')

mar['date'] = pd.to_datetime(mar['date'])
hlt['date'] = pd.to_datetime(hlt['date'])

mar_summer = mar[
    (mar['date'] >= '2025-06-01') &
    (mar['date'] <= '2025-08-15')
].copy()

hlt_summer = hlt[
    (hlt['date'] >= '2025-06-01') &
    (hlt['date'] <= '2025-08-15')
].copy()

abnb_corr = abnb_summer[['date', 'close']].copy()
abnb_corr['date'] = pd.to_datetime(abnb_corr['date'])
abnb_corr['close'] = pd.to_numeric(abnb_corr['close'], errors='coerce')
abnb_corr = abnb_corr.rename(columns={'close': 'Airbnb'})

mar_corr = mar_summer[['date', 'close']].copy()
mar_corr['date'] = pd.to_datetime(mar_corr['date'])
mar_corr['close'] = pd.to_numeric(mar_corr['close'], errors='coerce')
mar_corr = mar_corr.rename(columns={'close': 'Marriott'})

hlt_corr = hlt_summer[['date', 'close']].copy()
hlt_corr['date'] = pd.to_datetime(hlt_corr['date'])
hlt_corr['close'] = pd.to_numeric(hlt_corr['close'], errors='coerce')
hlt_corr = hlt_corr.rename(columns={'close': 'Hilton'})

tsa_corr = tsa_summer[['Date', 'Numbers']].copy()
tsa_corr['Date'] = pd.to_datetime(tsa_corr['Date'])
tsa_corr['Numbers'] = pd.to_numeric(
    tsa_corr['Numbers'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)
tsa_corr = tsa_corr.rename(columns={'Date': 'date', 'Numbers': 'TSA Passenger Volume'})

cpi_corr = cpi_summer[['observation_date', 'CUSR0000SEHB']].copy()
cpi_corr['observation_date'] = pd.to_datetime(cpi_corr['observation_date'])
cpi_corr['CUSR0000SEHB'] = pd.to_numeric(cpi_corr['CUSR0000SEHB'], errors='coerce')
cpi_corr = cpi_corr.rename(columns={'observation_date': 'date', 'CUSR0000SEHB': 'Lodging CPI'})

# Expand monthly CPI to daily values across the summer window for date alignment.
daily_range = pd.date_range(start='2025-06-01', end='2025-08-15', freq='D')
cpi_daily = (
    cpi_corr.set_index('date')
    .reindex(daily_range)
    .ffill()
    .rename_axis('date')
    .reset_index()
)

merged = (
    abnb_corr
    .merge(mar_corr, on='date', how='inner')
    .merge(hlt_corr, on='date', how='inner')
    .merge(tsa_corr, on='date', how='inner')
    .merge(cpi_daily, on='date', how='inner')
    .sort_values('date')
)

returns_df = merged[['Airbnb', 'Marriott', 'Hilton', 'TSA Passenger Volume', 'Lodging CPI']].pct_change().dropna()
correlation_matrix = returns_df.corr()

fig_corr = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    title='How Airbnb, Hotel Stocks, and Travel Indicators Relate (Summer 2025)'
)

fig_corr.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_title='Metrics',
    yaxis_title='Metrics',
    coloraxis_colorbar=dict(title='Correlation'),
    margin=dict(l=120, r=40, t=90, b=120)
)

fig_corr.update_xaxes(tickangle=-25, automargin=True, tickfont=dict(size=11))
fig_corr.update_yaxes(automargin=True, tickfont=dict(size=11))

fig_corr

In [ ]:
# Dash dashboard using existing figures with a shared date-range filter
from dash import Dash, dcc, html, Input, Output
import copy

start_date = pd.Timestamp('2025-06-01')
end_date = pd.Timestamp('2025-08-15')
all_dates = pd.date_range(start_date, end_date, freq='D')

# Use the existing figure objects created above (no chart recreation).
base_figures = {
    'tsa': fig_tsa_summer,
    'cpi': fig,
    'compare': fig_abnb_mar,
    'scatter': fig_abnb_tsa_ols,
}

def style_figure(fig_obj, *, height=360, showlegend=False, bottom_panel=False):
    panel_margin = dict(l=58, r=26, t=62, b=50) if bottom_panel else dict(l=52, r=20, t=64, b=44)
    fig_obj.update_layout(
        template='plotly_white',
        title_x=0.5,
        height=height,
        margin=panel_margin,
        showlegend=showlegend,
        font=dict(family='Arial, sans-serif', size=13, color='#222'),
        title_font=dict(size=18),
        xaxis_title_font=dict(size=13),
        yaxis_title_font=dict(size=13)
    )
    fig_obj.update_xaxes(automargin=True, tickfont=dict(size=11))
    fig_obj.update_yaxes(automargin=True, tickfont=dict(size=11))
    return fig_obj

def extract_date(interaction_data):
    if interaction_data and interaction_data.get('points'):
        x_value = interaction_data['points'][0].get('x')
        try:
            return pd.to_datetime(x_value).normalize()
        except Exception:
            return None
    return None

def highlight_points_by_date(fig_obj, target_date, selected_size=11, default_size=6):
    for trace in fig_obj.data:
        if not hasattr(trace, 'x') or trace.x is None:
            continue
        try:
            x_values = pd.to_datetime(list(trace.x))
        except Exception:
            continue
        marker_sizes = [selected_size if x.normalize() == target_date else default_size for x in x_values]
        trace.update(marker=dict(size=marker_sizes))

marks = {
    i: d.strftime('%b %d')
    for i, d in enumerate(all_dates)
    if i in (0, len(all_dates) - 1) or d.day in (1, 15)
}

app = Dash(__name__)

app.layout = html.Div(
    [
        html.H2(
            'Summer 2025 Travel & Market Dashboard',
            style={'textAlign': 'center', 'margin': '0 0 4px 0', 'fontFamily': 'Arial, sans-serif'}
        ),
        html.P(
            'Shared date filter updates all charts. Hover or click on TSA or Airbnb vs Marriott to sync date highlights.',
            style={'textAlign': 'center', 'margin': '0 0 8px 0', 'color': '#555', 'fontFamily': 'Arial, sans-serif'}
        ),
        dcc.RangeSlider(
            id='date-range',
            min=0,
            max=len(all_dates) - 1,
            value=[0, len(all_dates) - 1],
            marks=marks,
            allowCross=False,
            updatemode='mouseup',
            tooltip={'always_visible': False, 'placement': 'bottom'}
        ),
        html.Div(
            [
                dcc.Graph(id='chart-compare', style={'gridColumn': '1 / span 2'}),
                dcc.Graph(id='chart-tsa'),
                dcc.Graph(id='chart-cpi'),
                dcc.Graph(id='chart-scatter', style={'gridColumn': '1 / span 2'})
            ],
            style={
                'display': 'grid',
                'gridTemplateColumns': 'repeat(2, minmax(0, 1fr))',
                'gap': '10px',
                'marginTop': '10px',
                'alignItems': 'start'
            }
        )
    ],
    style={'maxWidth': '1320px', 'margin': '0 auto', 'padding': '10px 12px'}
)

@app.callback(
    Output('chart-compare', 'figure'),
    Output('chart-tsa', 'figure'),
    Output('chart-cpi', 'figure'),
    Output('chart-scatter', 'figure'),
    Input('date-range', 'value'),
    Input('chart-compare', 'hoverData'),
    Input('chart-tsa', 'hoverData'),
    Input('chart-compare', 'clickData'),
    Input('chart-tsa', 'clickData')
)
def update_all_charts(selected_range, compare_hover, tsa_hover, compare_click, tsa_click):
    start_idx, end_idx = selected_range
    start = all_dates[start_idx]
    end = all_dates[end_idx]

    compare_fig = style_figure(copy.deepcopy(base_figures['compare']), height=430, showlegend=True)
    tsa_fig = style_figure(copy.deepcopy(base_figures['tsa']), height=300, showlegend=False, bottom_panel=True)
    cpi_fig = style_figure(copy.deepcopy(base_figures['cpi']), height=300, showlegend=False, bottom_panel=True)
    scatter_fig = style_figure(copy.deepcopy(base_figures['scatter']), height=320, showlegend=True, bottom_panel=True)

    # Keep marker points visible for linked highlighting in time-series charts.
    compare_fig.update_traces(mode='lines+markers', marker=dict(size=6, opacity=0.9))
    tsa_fig.update_traces(mode='lines+markers', marker=dict(size=5, opacity=0.85))

    # Shared date window for charts with date-based x-axes.
    compare_fig.update_xaxes(range=[start, end])

    # Add slight horizontal padding to prevent edge points from being visually clipped.
    padded_start = start - pd.Timedelta(days=1)
    padded_end = end + pd.Timedelta(days=1)
    tsa_fig.update_xaxes(range=[padded_start, padded_end])
    cpi_fig.update_xaxes(range=[padded_start, padded_end])

    # Ensure CPI markers are not clipped at plot boundaries.
    cpi_fig.update_traces(cliponaxis=False)

    # Shared hover/click date linking between TSA and comparison chart.
    selected_date = (
        extract_date(compare_click)
        or extract_date(tsa_click)
        or extract_date(compare_hover)
        or extract_date(tsa_hover)
    )

    if selected_date is not None:
        for fig_obj in (compare_fig, tsa_fig, cpi_fig):
            fig_obj.add_vline(
                x=selected_date,
                line_width=1.5,
                line_dash='dot',
                line_color='rgba(70,70,70,0.55)'
            )

        highlight_points_by_date(compare_fig, selected_date, selected_size=11, default_size=6)
        highlight_points_by_date(tsa_fig, selected_date, selected_size=10, default_size=5)

    return compare_fig, tsa_fig, cpi_fig, scatter_fig

try:
    app.run(jupyter_mode='inline', jupyter_height=1220, debug=False)
except TypeError:
    app.run_server(debug=False, port=8050)